# 03 Inference and 3D Visualization

In [1]:
from pathlib import Path
import sys

_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve().parent.parent]
_PROJECT_ROOT = next((p for p in _candidates if (p / 'src' / 'solsys_emulator').exists()), None)
if _PROJECT_ROOT is None:
    raise RuntimeError('Impossibile trovare la project root con src/solsys_emulator')

_SRC = _PROJECT_ROOT / 'src'
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

print('Project root:', _PROJECT_ROOT)
print('Python executable:', sys.executable)



Project root: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system
Python executable: /Applications/Xcode.app/Contents/Developer/usr/bin/python3


In [2]:
import numpy as np

from solsys_emulator.config import DEFAULT_CHECKPOINT_PATH, DEFAULT_KERNEL_PATH
from solsys_emulator.de440_dataset import find_local_kernel, load_kernel, sample_states
from solsys_emulator.inference import EphemerisEmulator
from solsys_emulator.train import load_checkpoint
from solsys_emulator.time_frames import build_time_grid
from solsys_emulator.viz_3d import plot_scene

emu = EphemerisEmulator.from_checkpoint(DEFAULT_CHECKPOINT_PATH)

ckpt_meta = load_checkpoint(DEFAULT_CHECKPOINT_PATH)
train_meta = ckpt_meta.get('train_config', {})
print('Checkpoint split_mode:', train_meta.get('split_mode'))
print('Checkpoint nbody_loss_weight:', train_meta.get('nbody_loss_weight'))

# Tutti i corpi dinamici nel checkpoint (esclude il Sole dalle traiettorie)
bodies_to_compare = [b for b in emu.bodies if b.lower() != 'sun']
print('Corpi in confronto:', bodies_to_compare)

dt_plot_seconds = 21600.0  # 6 ore
plot_t0 = '2025-01-01T00:00:00'
plot_t1 = '2026-01-01T00:00:00'

# Traiettorie PINN
pred_traj = {
    body: emu.predict_trajectory(body, plot_t0, plot_t1, dt_plot_seconds)
    for body in bodies_to_compare
}

# Traiettorie vere da DE440/DE441 (strict)
kernel_path = find_local_kernel([DEFAULT_KERNEL_PATH])
if kernel_path is None:
    raise RuntimeError('Nessun kernel DE440/DE441 trovato in data/.')

kernel = load_kernel(kernel_path)
_, time_grid = build_time_grid(plot_t0, plot_t1, dt_plot_seconds)
true_states, meta = sample_states(time_grid, bodies=bodies_to_compare, kernel=kernel)
true_traj = {body: true_states[:, idx, :3] for idx, body in enumerate(bodies_to_compare)}

print('Kernel usato:', kernel_path)
print('Source:', meta.get('source'))

for body in bodies_to_compare:
    err = pred_traj[body] - true_traj[body]
    rmse = np.sqrt(np.mean(np.sum(err * err, axis=1)))
    print(f'{body:8s} RMSE position [km]: {rmse:,.2f}')

state_t = emu.predict_state('2025-06-01T00:00:00')
fig = plot_scene(
    states_at_t=state_t,
    trajectories=pred_traj,
    reference_trajectories=true_traj,
    reference_label='DE440',
    frame_label='barycentric icrs',
    units_label='km',
)
fig.show()


Checkpoint split_mode: random
Checkpoint nbody_loss_weight: 0.0
Corpi in confronto: ['mercury', 'venus', 'earth', 'moon', 'mars', 'jupiter', 'saturn', 'uranus', 'neptune']
Kernel usato: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/data/de440.bsp
Source: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/data/de440.bsp
mercury  RMSE position [km]: 59,573.22
venus    RMSE position [km]: 59,000.86
earth    RMSE position [km]: 61,673.02
moon     RMSE position [km]: 63,422.21
mars     RMSE position [km]: 96,648.49
jupiter  RMSE position [km]: 329,864.50
saturn   RMSE position [km]: 570,555.49
uranus   RMSE position [km]: 402,254.42
neptune  RMSE position [km]: 321,075.97
